# Suite QRY — Queries

A predicate is a query: `select(predicate, variables)` streams its matches among a store's data (each symbol's extent:
what the store's singletons reach) for which its rule holds. `Queries.Scan(store)` makes any store queryable in memory,
planning each query from the rule's shape; `explain` shows the plan.

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Patterns import Constraints as C, Queries as Q
from mbse.Schemas.Framework import Bindings as Bd, Schemas as S, Stores
from support import Contact, Phone, Phones, book, listed, native, protocol_problems, raises

the, c, p = E.variable("the"), E.variable("c"), E.variable("p")


def predicate(rule, **symbols):
    return C.OfPredicate.Builder().name("Q").symbols(symbols or {"the": Contact}).rule(rule).create()


store = book()
Pager = S.OfObject.Builder().name("Pager").ref().properties(native("number")).relations(
    lambda r: r.name("owners").of(Phones).me("phone")).create()  # a pager may fill a phone's link too
store.register(Pager)
ann = store.Contact().name("ann").age(30).phones(lambda e: e.phone(store.Phone().number("1").create())).phones(
    lambda e: e.phone(store.Phone().number("2").create())).phones(lambda e: e.phone(store.Pager().number("p").create())).create()
bob = store.Contact().name("bob").phones(lambda e: e.phone(store.Phone().create())).create()
cy = store.Contact().name("cy").age(9).create()
listed(store, ann, bob, cy)
loose = store.Contact().name("dee").age(50).create()  # never listed: not the store's data
query = Q.Scan(store)


def names(matches):
    return [{symbol: getattr(value, "name", None) or value.number for symbol, value in match.items()} for match in matches]

## QRY-01 · A query streams the matches for which its rule holds

In [ ]:
adults = query.select(predicate(the.age.ge(18)))
assert iter(adults) is adults and next(adults) == {"the": ann}  # an iterator of matches, read as it is asked
assert list(adults) == []  # the rest: bob's age is unknown, cy is a child, dee is not in the extent
assert list(query.select(predicate(the.age.ge(18)), unknown=True)) == [{"the": ann}, {"the": bob}]
assert names(query.select(predicate(the.has("name")))) == [{"the": "ann"}, {"the": "bob"}, {"the": "cy"}]

## QRY-02 · Variables are bound once, and objects too

In [ ]:
assert list(query.select(predicate(the.age.ge(E.variable("min"))), {"min": 21})) == [{"the": ann}]
assert list(query.select(predicate(the.age.lt(E.variable("max")).and_(the.ne(E.variable("not_")))),
                         {"max": 40, "not_": cy})) == [{"the": ann}]  # an object has no literal: bound per match
assert len(list(query.select(predicate(E.variable("flag").or_(the.age.ge(99))), {"flag": True}))) == 3
assert query.explain(predicate(E.variable("flag").and_(the.has("name"))), {"flag": False}) == [
    "first 1 test", "the: scan Contact"]  # the variables decide the rule: false, tested before any symbol is bound
assert list(query.select(predicate(E.variable("flag").and_(the.has("name"))), {"flag": False})) == []  # decided first
assert query.explain(predicate(E.variable("go").and_(the.has("name"))), {"go": E.variable("x")}) == [
    "first 1 test", "the: scan Contact, then 1 test"]  # a variable bound to no literal is tested before any symbol

## QRY-03 · The extents are read when the first match is asked for

In [ ]:
pending = query.select(predicate(the.age.ge(40)))
listed(store, loose)  # listed after the query was made, before it was read
assert list(pending) == [{"the": loose}]

## QRY-04 · A query that cannot be right is refused when it is made

In [ ]:
with raises(ValueError, match="a query needs a predicate with at least one symbol"):
    query.select(C.OfPredicate.Builder().rule(True).create())
with raises(ValueError, match="symbol 'v' needs a named reference object schema"):
    query.select(predicate(True, v=S.OfObject.Builder().name("V").create()))
with raises(AttributeError, match="no schema registered as 'Elsewhere'"):
    query.select(predicate(True, x=S.OfObject.Builder().name("Elsewhere").ref().create()))
with raises(ValueError, match="'the' is a symbol; it is not a variable"):
    query.select(predicate(the.has("name")), {"the": ann})
with raises(ValueError, match="the predicate cannot be a query: "):
    query.select(predicate(the.age.ge(E.variable("n"))))
with raises(ValueError, match="'pow' is not a core operation"):
    query.select(predicate(E.operation("pow", the.age, 2).ge(1)))
numbers = query.select(predicate(the.age))  # a rule must give a bool, found as matches are read
with raises(TypeError, match="a predicate must be a bool, got int"):
    list(numbers)

## QRY-05 · Several symbols match the cross product, narrowed by hops through relations

In [ ]:
any_phone = E.quantifier("any", "e", E.operation("entries", c, "phones"), E.variable("e").phone.eq(p))
owned = predicate(any_phone.and_(p.has("number")), c=Contact, p=Phone)
assert query.explain(owned) == ["c: scan Contact", "p: c.phones to phone, any number, then 2 tests"]
assert names(query.select(owned)) == [{"c": "ann", "p": "1"}, {"c": "ann", "p": "2"}]  # ann's pager is not a phone
any_owner = E.quantifier("any", "e", E.operation("entries", p, "owners"), E.variable("e").owner.eq(c))
by_phone = predicate(any_owner.and_(c.age.ge(18)), c=Contact, p=Phone)
assert query.explain(by_phone) == ["p: scan Phone", "c: p.owners to owner, at most one, then 2 tests"]  # unique(owner)
assert names(query.select(by_phone)) == [{"c": "ann", "p": "1"}, {"c": "ann", "p": "2"}]
pairs = predicate(c.age.gt(the.age), the=Contact, c=Contact)  # no hop: the cross product, tested once both are bound
assert query.explain(pairs) == ["the: scan Contact", "c: scan Contact, then 1 test"]
assert names(query.select(pairs)) == [{"the": "ann", "c": "dee"}, {"the": "cy", "c": "ann"}, {"the": "cy", "c": "dee"}]
both = predicate(any_phone.and_(any_owner).and_(p.has("number")), c=Contact, p=Phone)  # hops both ways: either first
assert query.explain(both) == ["c: scan Contact", "p: c.phones to phone, any number, then 3 tests"]
assert names(query.select(both)) == [{"c": "ann", "p": "1"}, {"c": "ann", "p": "2"}]
narrow = predicate(the.has("name").and_(any_phone).and_(c.ne(the)), the=Contact, c=Contact, p=Phone)
assert query.explain(narrow) == ["the: scan Contact, then 1 test", "c: scan Contact, then 1 test",
                                 "p: c.phones to phone, any number, then 1 test"]
assert len(list(query.select(narrow))) == 9  # three named contacts, each with ann's two phones and bob's one, but itself

## QRY-06 · Only a hop through a relation narrows candidates

In [ ]:
def plan(rule):
    return query.explain(predicate(rule, c=Contact, p=Phone))


e = E.variable("e")
scans = ["c: scan Contact", "p: scan Phone, then 1 test"]
assert plan(E.quantifier("all", "e", E.operation("entries", c, "phones"), e.phone.eq(p))) == scans  # not any
assert plan(E.quantifier("any", "e", E.operation("count", c), e.phone.eq(p))) == scans  # not entries
assert plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), e.phone.ne(p))) == scans  # not eq
assert query.explain(predicate(E.quantifier("any", "e", E.operation("entries", E.variable("x"), "phones"), e.phone.eq(p)),
                             c=Contact, p=Phone), {"x": ann}) == scans  # from a variable, not a symbol
assert plan(E.quantifier("any", "e", E.operation("entries", c, "nope"), e.phone.eq(p))) == scans  # no such adjacency
assert plan(E.quantifier("any", "e", E.operation("entries", c, 1), e.phone.eq(p))) == scans  # not an adjacency's name
assert plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), e.label.eq(p))) == scans  # a property
assert plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), e.owner.eq(p))) == scans  # its own link
assert plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), e.phone.eq(c))) == [
    "c: scan Contact, then 1 test", "p: scan Phone"]  # back to itself: a test of c alone
assert plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), e.eq(p))) == scans  # not a link's target
assert plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), p.eq(e.phone))) == [
    "c: scan Contact", "p: c.phones to phone, any number, then 1 test"]  # either side of eq

## QRY-07 · Scan is a store, and select asks a queryable store or scans another

In [ ]:
assert protocol_problems(query, Q.QueryableStore) == [] and protocol_problems(store, Q.QueryableStore) == ["select"]
assert (query.schema("Contact"), query.registered("Listed"), query.name_of(Contact)) == (Contact, store.registered("Listed"), "Contact")
assert query.names() == store.names() and query.extent("Contact") == store.extent("Contact")
assert query.singleton("book.Directory") is store.singleton("book.Directory")
assert query.member(ann, "name") == "ann" and query.builder("Contact").name("e").create().name == "e"
assert query.builder("Contact", ann).update() is ann and query.store is store
scanned = Q.select(store, predicate(the.age.ge(18)))  # a plain store is scanned; the update above reordered its entries
assert sorted(m["the"].name for m in scanned) == ["ann", "dee"]


class Native:
    """A store that answers queries itself."""

    def select(self, predicate, variables=None, unknown=False):
        return iter([("asked", predicate.name, variables, unknown)])


assert list(Q.select(Native(), predicate(True), {"v": 1}, True)) == [("asked", "Q", {"v": 1}, True)]
Item = S.OfObject.Builder().name("Item").ref().singleton("the.Item").properties(native("name")).create()
binding = Bd.Binding(Item, lambda t: Bd.State({"name": t.name}),
                     lambda s: type("Thing", (), {"name": s.values.get("name"), "accept": lambda self, v: Bd.accept(binding, self, v),
                                                  "identity": lambda self: id(self), "schema_name": lambda self: "Item",
                                                  "owner": lambda self: None})())
bound = Bd.OfStore([(Item, lambda instance=None: Bd.Builder(binding, instance))])
things = Q.select(bound, predicate(the.has("name").not_(), the=Item))
assert list(things) == [{"the": bound.singleton("the.Item")}]  # bound classes too